### Setting up

In [ ]:
# Add SSAS dlls to path
from sys import path
path.append('./Microsoft.AnalysisServices.Tabular')
path.append('./Microsoft.AnalysisServices.AdomdClient')

# Import packages
import os
import requests
import urllib.parse
import shutil
import asyncio
import pandas as pd
from time import time
from typing import Dict
from auth import Auth
from workspace import Workspace
from dataflow import Dataflow
from dataset import Dataset
from pyadomd import Pyadomd

# Tenant/app settings
TENANT_ID = os.environ.get('TENANT_ID', '')
CLIENT_ID = os.environ.get('CLIENT_ID', '')
CLIENT_SECRET = os.environ.get('CLIENT_SECRET', '')

ROOT_PATH = './data/data_hub'

HOST = 'example.sharepoint.com'
SITE = 'example_site_name'
SITE_URL = f'https://{HOST}/sites/{SITE}'
relativeUrl = f'General/2. Arquivos/8. Data Hub'

In [ ]:
# If ROOT_PATH exists, removes it and create a fresh one.
if os.path.isdir(ROOT_PATH):
    shutil.rmtree(ROOT_PATH)

os.mkdir(ROOT_PATH)

In [ ]:
# Authentication (get bearer token)
auth = Auth(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
token = auth.get_token()

# Initializing objects
workspace = Workspace(token)
dataset = Dataset(token)
dataflow = Dataflow(token)

### Getting workspaces data

In [ ]:
# Ignore workspaces list
ignore_workspaces = [
    'Brewdat Dev - GHQ - People - PSI'
]

# Get list of workspaces
workspaces_list = workspace.list_workspaces()
workspaces_data= workspaces_list.get('content', [])
df_workspaces = pd.DataFrame(workspaces_data)

# Create columns
df_workspaces['premium_link'] = 'powerbi://api.powerbi.com/v1.0/myorg/' + df_workspaces['name']
df_workspaces['webUrl'] = 'https://app.powerbi.com/groups/' + df_workspaces['id']

# Remove without capacityId and from ignore list
df_workspaces = df_workspaces[(~df_workspaces['capacityId'].isna())&(~df_workspaces['name'].isin(ignore_workspaces))]

# Sort and save
df_workspaces.sort_values(by=['name'], inplace=True)
df_workspaces.reset_index(inplace=True, drop=True)
df_workspaces.to_csv(f'{ROOT_PATH}/workspaces.csv', sep='@', index=False, encoding='utf-8-sig')

print('Total columns:', len(df_workspaces.columns))
df_workspaces.describe()

### Getting datasets data

In [ ]:
# Ignore datasets and reports that are not used for our Data Hub
ignore_list = [
    'Report Usage Metrics Model',
    'Report Usage Metrics Report',
    'Usage Metrics Report',
    'Dashboard Usage Metrics Model',
    'Access Control Report'
]

# Get datasets data
df_datasets_list = []
for workspace_to_search in df_workspaces.to_dict('records'):

    datasets_list = dataset.list_datasets(workspace_id=workspace_to_search['id'])
    datasets_data = datasets_list.get('content', [])
    df = pd.DataFrame(datasets_data)
    df['workspace_id'] = workspace_to_search['id']
    df['workspace_name'] = workspace_to_search['name']
    df_datasets_list.append(df)

    df = None
    del df

df_datasets = pd.concat(df_datasets_list)

# Create columns
df_datasets['premium_link'] = 'powerbi://api.powerbi.com/v1.0/myorg/' + df_datasets['workspace_name']

# Remove untitled and from ignore list
df_datasets = df_datasets[(~df_datasets['name'].isin(ignore_list))&(~df_datasets['name'].str.contains('Untitled'))]

# Apply filters
df_datasets = df_datasets[df_datasets['addRowsAPIEnabled']==False]

# Sort and save
df_datasets.sort_values(by=['workspace_name', 'name'])
df_datasets.reset_index(inplace=True, drop=True)
df_datasets.to_csv(f'{ROOT_PATH}/datasets.csv', sep='@', index=False, encoding='utf-8-sig')

print('Total columns:', len(df_datasets.columns))
df_datasets.describe()

### Getting reports data

In [ ]:
# Get reports data
df_reports_list = []
for workspace_to_search in df_workspaces.to_dict('records'):

    reports_list = workspace.list_reports(workspace_id=workspace_to_search['id'])
    reports_data = reports_list.get('content', [])
    df = pd.DataFrame(reports_data)
    df['workspace_id'] = workspace_to_search['id']
    df['workspace_name'] = workspace_to_search['name']
    df_reports_list.append(df)

    df = None
    del df

df_reports = pd.concat(df_reports_list)

# Apply filters
df_reports = df_reports[(~df_reports['name'].isin(ignore_list))&(~df_reports['name'].str.contains('Untitled'))]

# Sort and save
df_reports.sort_values(by=['workspace_name', 'name'])
df_reports.to_csv(f'{ROOT_PATH}/reports.csv', sep='@', index=False, encoding='utf-8-sig')

print('Total columns:', len(df_reports.columns))
df_reports.describe()

### Getting tables, columns and measures data
##### We get this connecting to the SSAS model for each dataset using XMLA endpoints.

In [22]:
async def fetch_data(workspace_name: str, dataset_id: str, dataset_name: str, token: str) -> Dict:
    """
    Get data from a PBI Dataset:
        - Tables;
        - Columns;
        - Measures;
        - Model;

    Args:
        workspace_name (str): workspace name.
        dataset_id (str): dataset id.
        dataset_name (str): dataset name
        token (str): token for authentication

    Returns:
        data (dict): dictionary with all the data for that dataset.
    """
    data = {}
                
    data['workspace_name'] = workspace_name
    data['dataset_id'] = dataset_id
    data['dataset_name'] = dataset_name
    
    workspace_path = f'{ROOT_PATH}/{workspace_name}'
    dataset_path = f'{workspace_path}/{dataset_name}'

    # Create folders if they do not exist
    for p in [workspace_path, dataset_path]:
        if not os.path.exists(p):
            os.mkdir(p)

    # Build the connection string
    connection_string_part1 = f'Provider=MSOLAP.8;Data Source=powerbi://api.powerbi.com/v1.0/myorg/{workspace_name};Initial Catalog={dataset_name};'
    connection_string_part2 = f'User ID=;Password={token};'
    connection_string = connection_string_part1 + connection_string_part2
    
    # If we don't have the files already.
    # This is just for testing, in production this should be removed.
    if  (
              (not os.path.isfile(f'{dataset_path}/tables.csv'))
            | (not os.path.isfile(f'{dataset_path}/columns.csv'))
            | (not os.path.isfile(f'{dataset_path}/measures.csv'))
            | (not os.path.isfile(f'{dataset_path}/models.csv'))
        ):
            
            try:

                with Pyadomd(connection_string) as conn:

                    # Get tables data
                    tables_query = ''' 
                        SELECT [ID] AS [Table ID] ,
                            [Name] AS [Table] ,
                            [IsHidden] ,
                            [IsPrivate] ,
                            [ModifiedTime] AS [Modified Datetime] ,
                            [StructureModifiedTime] AS [Structure Modified Datetime]
                        FROM $SYSTEM.TMSCHEMA_TABLES
                    '''
                    with conn.cursor().execute(tables_query) as tables:

                        tables_data = tables.fetchall()
                        data['tables_header'] = [c.name for c in tables.description]
                        data['tables_data'] = tables_data
                        df = pd.DataFrame(data['tables_data'], columns=data['tables_header'])
                        df['workspace_name'] = data['workspace_name']
                        df['dataset_name'] = data['dataset_name']
                        df['dataset_id'] = data['dataset_id']
                        df.to_csv(f'{dataset_path}/tables.csv', sep='@', index=False, encoding='utf-8-sig')

                        del df

                    # Get columns data
                    columns_query = '''
                        SELECT [ID] AS [Column ID],
                            [TableID] AS [Table ID],
                            [ExplicitName] AS [Column],
                            [InferredName] AS [Column Inferred Name],
                            [ExplicitDataType] AS [Data Type ID],
                            [InferredDataType] AS [Inferred Data Type ID],
                            [DataCategory] AS [Data Category],
                            [SourceColumn] AS [Source Column]
                        FROM $SYSTEM.TMSCHEMA_COLUMNS
                    '''
                    with conn.cursor().execute(columns_query) as columns:
                        columns_data = columns.fetchall()
                        data['columns_header'] = [c.name for c in columns.description]
                        data['columns_data'] = columns_data
                        df = pd.DataFrame(data['columns_data'], columns=data['columns_header'])
                        df['workspace_name'] = data['workspace_name']
                        df['dataset_name'] = data['dataset_name']
                        df['dataset_id'] = data['dataset_id']
                        df.to_csv(f'{dataset_path}/columns.csv', sep='@', index=False, encoding='utf-8-sig')

                        del df

                    # Get measures data
                    measures_query = '''
                        SELECT [ID] AS [Measure ID],
                        [TableID] AS [Table ID],
                        [Name],
                        [DataType] AS [Data Type ID],
                        [DisplayFolder] AS [Display Folder],
                        [Description],
                        [Expression]
                        FROM $SYSTEM.TMSCHEMA_MEASURES
                    '''
                    with conn.cursor().execute(measures_query) as measures:
                        measures_data = measures.fetchall()
                        data['measures_header'] = [c.name for c in measures.description]
                        data['measures_data'] = measures_data
                        df = pd.DataFrame(data['measures_data'], columns=data['measures_header'])
                        df['workspace_name'] = data['workspace_name']
                        df['dataset_name'] = data['dataset_name']
                        df['dataset_id'] = data['dataset_id']
                        df.to_csv(f'{dataset_path}/measures.csv', sep='@', index=False, encoding='utf-8-sig')

                        del df

                # Get models data
                with Pyadomd(connection_string) as conn:
                    models_query = '''
                        SELECT *
                        FROM $SYSTEM.TMSCHEMA_MODEL
                    '''
                    with conn.cursor().execute(models_query) as models:
                        models_data = models.fetchall()
                        data['models_header'] = [c.name for c in models.description]
                        data['models_data'] = models_data
                        df = pd.DataFrame(data['models_data'], columns=data['models_header'])
                        df['workspace_name'] = data['workspace_name']
                        df['dataset_name'] = data['dataset_name']
                        df['dataset_id'] = data['dataset_id']
                        df.to_csv(f'{dataset_path}/models.csv', sep='@', index=False, encoding='utf-8-sig')

                        del df

            except Exception as e:
                print(f'Error with {workspace_name} - {dataset_id} - {dataset_name}')

    return data


async def main() -> Dict:
    """
    This is the main function and returns all the data from all datasets:
        - Dataset name and id;
        - Tables;
        - Columns;
        - Measures;
        - Model;

    Returns:
        results (dict): dictionary with data from all datasets.
    """

    # Token for authentication
    refresh_token_every_x_seconds = 600
    token_refresh_time = time()
    token = auth.get_token()
    
    datasets_id = df_datasets['id'].values.tolist()
    datasets_name = df_datasets['name'].values.tolist()
    workspaces_names = df_datasets['workspace_name'].tolist()


    tasks = []
    for workspace_name, dataset_id, dataset_name in zip(workspaces_names, datasets_id, datasets_name):

        # Refresh token periodically
        time_since_last_token_refresh = time() - token_refresh_time
        if time_since_last_token_refresh > refresh_token_every_x_seconds:
            print('Refreshing token...')
            token = auth.get_token()

        tasks.append(asyncio.ensure_future(fetch_data(workspace_name, dataset_id, dataset_name, token)))

    results = await asyncio.gather(*tasks)

    return results


results = await main()

### Send to SharePoint

In [ ]:
# Authentication (get bearer token)
auth = Auth(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
token = auth.get_token_for_sharepoint(HOST)

file_local_path = f'{ROOT_PATH}/workspaces.csv'
with open(file_local_path, 'rb') as file:
    file_content = file.read()

file_name = os.path.basename(file_local_path)

headers = {
    'Authorization': f'Bearer {token}',
    'Content-Length': str(len(file_content)),
    'Accept': 'application/json; odata=verbose'
}

url = f"{SITE_URL}/_api/web/GetFolderByServerRelativeUrl('/Shared Documents/{urllib.parse.quote(relativeUrl)}')/Files/add(url='{file_name}',overwrite=true)"
r = requests.post(url, data=file_content, headers=headers)